# Global Olympic Sports Performance Analysis (1896 – 2016)
### A Comprehensive Empirical Analytics Study Powered by Python, Pandas & Tableau Public

**Author**: Md. Firdous Alam  
**Course**: Skills Wallet Capstone Project | Data Analytics & Business Intelligence  
**GitHub Repository**: [Olympic-Sports-Analysis](https://github.com/firdousalam2006vk/Olympic-Sports-Analysis)  
**Live Tableau Public Dashboard**: [Interactive Olympic Analytics](https://public.tableau.com/app/profile/md.firdous.alam/viz/GlobalOlympicSportsPerformanceAnalysis_17933559199720/Dashboard3)

---

## 1. Executive Overview & Problem Statement
Traditional sports datasets often log individual athletes participating in team disciplines (e.g., a 12-person basketball roster or 22-person soccer team). When analysts sum raw athlete records, team sports award a medal to every roster athlete, inflating national totals by over **300% to 1100%** (e.g. inflating USA's medal count past 5,600 medals).

This study solves the **Team Multi-Counting Trap** by engineering deterministic event deduplication across `[Year, Season, NOC, Sport, Event, Medal]`. This aligns programmatic totals exactly with official **International Olympic Committee (IOC)** records (USA: 2,823 official event medals).

In [1]:
# Import core data processing and visualization libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic visualization standards
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

print("Libraries successfully imported. Ready for analysis.")

## 2. Dataset Loading & Relational Schema Inspection
We utilize two primary structured datasets:
1. `data/medal_results.csv`: 3,837 rows covering all 51 Olympic editions (1896–2016) with zero duplicate keys.
2. `data/athletes.csv`: 157,417 modern Olympians (1980–2016) optimized to 22.78 MB (<25 MB GitHub upload limit).

In [2]:
# Load official deduplicated medal results and athlete biometrics
df_medals = pd.read_csv("../data/medal_results.csv")
df_athletes = pd.read_csv("../data/athletes.csv", low_memory=False)

print(f"Medal Results Dataset : {df_medals.shape[0]:,} rows x {df_medals.shape[1]} columns")
print(f"Athlete Records Dataset: {df_athletes.shape[0]:,} rows x {df_athletes.shape[1]} columns")
display(df_medals.head(3))

## 3. All-Time National Medal Leaderboard (Top 10 Nations)
Computing the exact deduplicated event medal count for all participating National Olympic Committees (NOCs).

In [3]:
# Aggregate total medals by nation
top_nations = df_medals.groupby(['noc', 'country'])[['gold', 'silver', 'bronze', 'total_medals']].sum()
top_10 = top_nations.sort_values(by='total_medals', ascending=False).head(10)
display(top_10)

## 4. Historical Participation & Gender Parity Evolution
Tracking female participation growth across the modern era (1980 to 2016).

In [4]:
# Calculate gender participation ratio per edition
gender_counts = df_athletes.groupby(['year', 'sex'])['athlete_id'].nunique().unstack(fill_value=0)
gender_counts['Total'] = gender_counts['M'] + gender_counts['F']
gender_counts['Female_Pct'] = (gender_counts['F'] / gender_counts['Total']) * 100

print("Female Participation Trajectory (1980 - 2016):")
display(gender_counts[['F', 'M', 'Total', 'Female_Pct']])

## 5. Host Nation Advantage Analysis
Evaluating whether hosting an Olympic Games delivers a statistically significant performance boost compared to historical baselines.

In [5]:
# Load processed host nation performance dataset
df_host = pd.read_csv("../data/processed/host_nation_performance.csv")
valid_hosts = df_host.dropna(subset=['medal_bump'])

surge_rate = (valid_hosts['medal_bump'] > 0).mean() * 100
mean_bump = valid_hosts['medal_bump'].mean()

print(f"Evaluated Host Editions : {len(valid_hosts)}")
print(f"Positive Surge Rate     : {surge_rate:.1f}%")
print(f"Average Net Medal Bump  : +{mean_bump:.1f} medals")
display(valid_hosts[['host_year', 'country', 'pre_medals', 'host_medals', 'medal_bump', 'medal_bump_pct']])

## 6. High-Performance Sports Economics & Investment ROI
Investigating the relationship between quadrennial public funding and Olympic medal success using verified UK Sport figures.

In [6]:
# Load sports funding benchmark
df_fund = pd.read_csv("../data/processed/sports_funding_benchmark.csv")
fund_col = [c for c in df_fund.columns if 'fund' in c.lower() or 'invest' in c.lower()][0]
med_col = [c for c in df_fund.columns if 'medal' in c.lower()][0]

r = df_fund[fund_col].corr(df_fund[med_col])
print(f"Pearson Correlation (Funding vs. Medals Won): r = {r:.3f}")
display(df_fund.head(5))

## 7. Strategic Conclusions & Tableau Integration
1. **Deduplication Solved**: Raw athlete inflation removed; 2,823 USA medals verified against IOC records.
2. **Host Advantage Proven**: 86.7% of host nations gain an average of +15.4 medals.
3. **Gender Parity Revolution**: Female participation surged from 21.5% in 1980 to 45.0% in 2016.
4. **Interactive Dashboards**: All aggregated findings are integrated into our live cloud-hosted Tableau Public workbook.